# LeetCode #150 — Evaluate Reverse Polish Notation: tính biểu thức hậu tố bằng ngăn xếp

**Mức độ:** Medium · **Chủ đề:** Array, Math, Stack

Skeleton trên LeetCode:

```python
class Solution:
    def evalRPN(self, tokens: List[str]) -> int:
```

Mục tiêu của notebook:

1. Hiểu **ký pháp Ba Lan ngược (RPN / hậu tố)**: toán tử đứng **sau** hai toán hạng, không cần ngoặc.
2. Thuật toán **ngăn xếp**: gặp số → push; gặp toán tử → pop 2, tính, push.
3. Hai chi tiết dễ sai: **thứ tự toán hạng** (`a − b` chứ không phải `b − a`) và **chia lấy phần nguyên làm tròn về 0** (khác `//` của Python với số âm).

# 1. Bài toán là gì?

Cho mảng `tokens` biểu diễn một biểu thức số học dạng **RPN**. Tính giá trị (số nguyên).

- Toán tử hợp lệ: `'+'`, `'-'`, `'*'`, `'/'`.
- Mỗi toán hạng là số nguyên hoặc kết quả của biểu thức khác.
- Phép chia giữa hai số nguyên **luôn làm tròn về 0** (truncate toward zero).
- Không có phép chia cho 0; biểu thức luôn hợp lệ; kết quả và mọi giá trị trung gian vừa số nguyên 32-bit.

**Constraints (theo đề):**

- `1 <= tokens.length <= 10^4`
- `tokens[i]` là một toán tử hoặc một số nguyên trong `[-200, 200]`.

---

## Ví dụ

```text
["2","1","+","3","*"]                                     ->  9     ((2 + 1) * 3)
["4","13","5","/","+"]                                    ->  6     (4 + (13 / 5))
["10","6","9","3","+","-11","*","/","*","17","+","5","+"] ->  22
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ngăn xếp (stack)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ngăn xếp (stack)

### Ngăn xếp là gì?

Ngăn xếp là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out), giống chồng đĩa: chỉ đặt thêm hoặc lấy ra ở **đỉnh**.

```text
push 1   push 2   push 3   pop → 3   top = 2
                   │ 3 │
          │ 2 │    │ 2 │    │ 2 │
 │ 1 │    │ 1 │    │ 1 │    │ 1 │
 └───┘    └───┘    └───┘    └───┘
```

| Thao tác | Ý nghĩa | Python (`list`) | Độ phức tạp |
|---|---|---|---|
| `push(x)` | đặt `x` lên đỉnh | `st.append(x)` | `O(1)` |
| `pop()` | lấy và bỏ đỉnh | `st.pop()` | `O(1)` |
| `top()` | xem đỉnh | `st[-1]` | `O(1)` |
| rỗng? | | `not st` | `O(1)` |

In [1]:
st = []
for x in [1, 2, 3]:
    st.append(x)
    print("push", x, "->", st)
print("top:", st[-1])
while st:
    print("pop", st.pop(), "->", st)
try:
    st.pop()
except IndexError as e:
    print("pop khi rỗng:", e)

push 1 -> [1]
push 2 -> [1, 2]
push 3 -> [1, 2, 3]
top: 3
pop 3 -> [1, 2]
pop 2 -> [1]
pop 1 -> []
pop khi rỗng: pop from empty list


### Khi nào nghĩ tới ngăn xếp?

Khi thứ được mở **sau cùng** phải được đóng/xử lý **trước tiên**:

- Dấu ngoặc lồng nhau: `( [ { } ] )`.
- Đường dẫn thư mục: `..` hủy thư mục vừa vào.
- Tính biểu thức hậu tố, gọi hàm đệ quy (máy tính dùng *call stack*).
- "Hoàn tác" (undo).

Ví dụ kinh điển — kiểm tra ngoặc hợp lệ: gặp ngoặc mở thì push; gặp ngoặc đóng thì đỉnh phải là ngoặc mở tương ứng.

In [2]:
def ngoac_hop_le(s):
    cap = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        else:
            if not st or st[-1] != cap[ch]:
                return False
            st.pop()
    return not st                        # còn ngoặc mở chưa đóng → sai

for s in ["([]{})", "([)]", "((", "}"]:
    print(f"{s!r:>9}: {ngoac_hop_le(s)}")

 '([]{})': True
   '([)]': False
     '((': False
      '}': False


### Lỗi hay gặp

- `pop()` / `st[-1]` khi ngăn xếp rỗng → `IndexError`. Luôn kiểm tra `if st`.
- Quên kiểm tra ngăn xếp **còn sót** phần tử ở cuối.
- Dùng `list.pop(0)` / `insert(0, x)` (đầu danh sách) — đó là hàng đợi và tốn `O(n)`; ngăn xếp luôn thao tác ở **cuối** list.

## 2.2 Áp dụng vào bài này

Biểu thức hậu tố tính bằng **ngăn xếp**: gặp số thì push; gặp toán tử thì pop hai số (số pop **trước** là toán hạng **phải**), tính và push kết quả.

# 3. Nền tảng Python cần biết

## Ngăn xếp bằng list

In [3]:
from typing import List
import random, time, operator

stack = []
stack.append(3); stack.append(5)
print("pop:", stack.pop(), "| còn:", stack)

pop: 5 | còn: [3]


## Chia làm tròn về 0 — **không** dùng `//`

`//` của Python làm tròn **xuống** (về `−∞`). Với kết quả âm thì khác làm tròn về 0:

In [4]:
for a, b in [(13, 5), (-13, 5), (13, -5), (-13, -5), (6, -132)]:
    print(f"{a:>4} / {b:>4}: a // b = {a // b:>3}  |  int(a / b) = {int(a / b):>3}")

  13 /    5: a // b =   2  |  int(a / b) =   2
 -13 /    5: a // b =  -3  |  int(a / b) =  -2
  13 /   -5: a // b =  -3  |  int(a / b) =  -2
 -13 /   -5: a // b =   2  |  int(a / b) =   2
   6 / -132: a // b =  -1  |  int(a / b) =   0


`int(a / b)` cắt phần thập phân → làm tròn về 0. (Với số lớn, `a / b` qua float có thể mất chính xác; ở đây giá trị vừa 32-bit nên an toàn. Cách thuần số nguyên: `q = abs(a) // abs(b)`, đổi dấu nếu `a`, `b` trái dấu.)

In [5]:
def div_trunc(a: int, b: int) -> int:
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

for a, b in [(13, 5), (-13, 5), (13, -5), (-13, -5), (6, -132), (0, -3)]:
    assert div_trunc(a, b) == int(a / b)
print("div_trunc khớp int(a / b)")

div_trunc khớp int(a / b)


## Nhận diện token là số

`"-11"` là **số âm**, không phải toán tử `-`. So sánh `token in "+-*/"` chỉ đúng khi token đúng **một ký tự**; an toàn nhất là kiểm tra bằng một tập toán tử.

# 4. Hậu tố là gì?

| Trung tố (infix) | Hậu tố (RPN) |
|---|---|
| `(2 + 1) * 3` | `2 1 + 3 *` |
| `4 + 13 / 5` | `4 13 5 / +` |
| `a - b` | `a b -` |

Đọc từ trái sang phải: gặp toán tử thì nó áp dụng lên **hai giá trị gần nhất** chưa được dùng — đúng tính chất LIFO của ngăn xếp. Không cần ngoặc hay độ ưu tiên.

# 5. Ý tưởng chính

```text
stack = []
for t in tokens:
    if t là toán tử:
        b = stack.pop()          # toán hạng PHẢI (đẩy vào sau)
        a = stack.pop()          # toán hạng TRÁI
        stack.append(a op b)
    else:
        stack.append(int(t))
return stack[0]
```

In [6]:
OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": div_trunc,
}


def eval_rpn(tokens: List[str]) -> int:
    stack = []
    for t in tokens:
        if t in OPS:
            b = stack.pop()
            a = stack.pop()
            stack.append(OPS[t](a, b))
        else:
            stack.append(int(t))
    return stack[0]

print(eval_rpn(["2", "1", "+", "3", "*"]), eval_rpn(["4", "13", "5", "/", "+"]),
      eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]))

9 6 22


# 6. Mô phỏng

In [7]:
def trace_rpn(tokens):
    stack = []
    print(f"{'token':>6} | stack")
    for t in tokens:
        if t in OPS:
            b = stack.pop(); a = stack.pop()
            r = OPS[t](a, b)
            stack.append(r)
            note = f"   ({a} {t} {b} = {r})"
        else:
            stack.append(int(t))
            note = ""
        print(f"{t:>6} | {stack}{note}")
    return stack[0]

_ = trace_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"])

 token | stack
    10 | [10]
     6 | [10, 6]
     9 | [10, 6, 9]
     3 | [10, 6, 9, 3]
     + | [10, 6, 12]   (9 + 3 = 12)
   -11 | [10, 6, 12, -11]
     * | [10, 6, -132]   (12 * -11 = -132)
     / | [10, 0]   (6 / -132 = 0)
     * | [0]   (10 * 0 = 0)
    17 | [0, 17]
     + | [17]   (0 + 17 = 17)
     5 | [17, 5]
     + | [22]   (17 + 5 = 22)


Chú ý bước `6 / -132`: kết quả `−0.045…` làm tròn về 0 thành `0` (còn `6 // -132 = −1`).

# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def evalRPN(self, tokens: List[str]) -> int:
        stack = []
        for t in tokens:
            if t in ("+", "-", "*", "/"):
                b = stack.pop()
                a = stack.pop()
                if t == "+":
                    stack.append(a + b)
                elif t == "-":
                    stack.append(a - b)
                elif t == "*":
                    stack.append(a * b)
                else:
                    stack.append(int(a / b))
            else:
                stack.append(int(t))
        return stack[0]

Để kiểm tra, ta viết bộ sinh biểu thức ngẫu nhiên: sinh **cây biểu thức**, in ra dạng hậu tố, và tính giá trị trực tiếp trên cây (đệ quy) làm đáp án.

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def random_expr(depth):
    """Trả về (tokens, value). Tránh chia cho 0."""
    if depth == 0 or random.random() < 0.3:
        v = random.randint(-200, 200)
        return [str(v)], v
    lt, lv = random_expr(depth - 1)
    rt, rv = random_expr(depth - 1)
    op = random.choice("+-*/")
    if op == "/" and rv == 0:
        op = "+"
    val = {"+": lv + rv, "-": lv - rv, "*": lv * rv, "/": div_trunc(lv, rv) if rv else 0}[op]
    return lt + rt + [op], val


sol = Solution()
check("Ví dụ 1", sol.evalRPN(["2", "1", "+", "3", "*"]), 9)
check("Ví dụ 2", sol.evalRPN(["4", "13", "5", "/", "+"]), 6)
check("Ví dụ 3", sol.evalRPN(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]), 22)
check("Một số", sol.evalRPN(["-7"]), -7)
check("Chia âm", sol.evalRPN(["-7", "2", "/"]), -3)
check("Thứ tự trừ", sol.evalRPN(["3", "10", "-"]), -7)

random.seed(150)
for _ in range(5000):
    toks, val = random_expr(random.randint(0, 6))
    assert sol.evalRPN(toks) == val == eval_rpn(toks), (toks, val)
print("5000 biểu thức ngẫu nhiên: khớp giá trị tính trên cây")

toks = ["1"]
for i in range(4999):
    toks += [str(random.randint(1, 200)), random.choice("+-")]
t0 = time.time(); r = sol.evalRPN(toks); t1 = time.time()
print(f"{len(toks)} token: {t1 - t0:.4f}s")

Ví dụ 1          PASS  9
Ví dụ 2          PASS  6
Ví dụ 3          PASS  22
Một số           PASS  -7
Chia âm          PASS  -3
Thứ tự trừ       PASS  -7


5000 biểu thức ngẫu nhiên: khớp giá trị tính trên cây


9999 token: 0.0061s


# 8. Vì sao đúng? Độ phức tạp?

**Quy nạp theo cấu trúc biểu thức:** một biểu thức hậu tố hợp lệ `E` có dạng `số` hoặc `E1 E2 op`. Khẳng định: đọc hết `E` thì ngăn xếp có thêm **đúng một** giá trị — giá trị của `E` — nằm trên phần cũ không đổi.

- `số`: push một giá trị. ✓
- `E1 E2 op`: sau `E1` đỉnh là `v1`; sau `E2` đỉnh là `v2`, ngay dưới là `v1`; `op` pop `v2` (= `b`) rồi `v1` (= `a`), push `v1 op v2`. ✓

```text
Time:  O(n) — mỗi token push/pop O(1)
Space: O(n) — ngăn xếp (tối đa khoảng n/2 + 1 số)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Đảo thứ tự toán hạng

In [10]:
def wrong_order(tokens):
    stack = []
    for t in tokens:
        if t in OPS:
            a = stack.pop()                        # SAI: pop đầu tiên là toán hạng PHẢI
            b = stack.pop()
            stack.append(OPS[t](a, b))
        else:
            stack.append(int(t))
    return stack[0]

print("Sai :", wrong_order(["4", "13", "5", "/", "+"]), "| Đúng:", eval_rpn(["4", "13", "5", "/", "+"]))
print("Sai :", wrong_order(["3", "10", "-"]), "| Đúng:", eval_rpn(["3", "10", "-"]))

Sai : 4 | Đúng: 6
Sai : 7 | Đúng: -7


`+` và `*` giao hoán nên không lộ lỗi; `-` và `/` thì sai ngay.

## Lỗi 2: Dùng `//` cho phép chia

In [11]:
def wrong_floor(tokens):
    stack = []
    for t in tokens:
        if t in OPS:
            b = stack.pop(); a = stack.pop()
            stack.append(a // b if t == "/" else OPS[t](a, b))   # SAI với kết quả âm
        else:
            stack.append(int(t))
    return stack[0]

ex3 = ["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]
print("Sai :", wrong_floor(ex3), "| Đúng:", eval_rpn(ex3))

Sai : 12 | Đúng: 22


`6 // -132 = -1` thay vì `0` → `10 * -1 + 17 + 5 = 12` thay vì `22`.

## Lỗi 3: Nhận diện toán tử bằng `t[0]` hoặc `t.isdigit()`

In [12]:
def wrong_detect(tokens):
    stack = []
    for t in tokens:
        if not t.isdigit():                        # SAI: "-11".isdigit() là False
            b = stack.pop(); a = stack.pop()
            stack.append(OPS[t](a, b))
        else:
            stack.append(int(t))
    return stack[0]

try:
    print(wrong_detect(ex3))
except KeyError as e:
    print("KeyError:", e, "— số âm bị coi là toán tử")

KeyError: '-11' — số âm bị coi là toán tử


## Lỗi 4: Dùng `eval(" ".join(...))` hay chuyển sang trung tố

Chậm, không an toàn với dữ liệu ngoài, và vẫn dính lỗi chia `//` hoặc `/` (ra float).

# 10. Mẹo nhận diện pattern

- **Biểu thức hậu tố** → một ngăn xếp giá trị.
- **Biểu thức trung tố có ngoặc / độ ưu tiên** → hai ngăn xếp (toán hạng + toán tử) hoặc thuật toán **shunting-yard** đổi sang hậu tố trước.
- **"Toán tử áp dụng lên các giá trị gần nhất"** → LIFO.

Bài liên quan: **#224 Basic Calculator** (Hard), **#227 Basic Calculator II**, **#772 Basic Calculator III**, **#20 Valid Parentheses**, **#1628 Design an Expression Tree With Evaluate Function**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `["5","1","2","+","4","*","+","3","-"]` và ghi trạng thái ngăn xếp sau mỗi token.

## Bài 2 — Điền chỗ trống

In [13]:
def exercise_2_eval_rpn(tokens: List[str]) -> int:
    stack = []
    for t in tokens:
        # TODO: toán tử → pop b, pop a, push a op b (chia làm tròn về 0); số → push
        pass
    return stack[0]


# assert exercise_2_eval_rpn(["2", "1", "+", "3", "*"]) == 9
# assert exercise_2_eval_rpn(["4", "13", "5", "/", "+"]) == 6
# assert exercise_2_eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22

## Bài 3 — Biến thể: trung tố → hậu tố (shunting-yard)

Chuyển biểu thức trung tố gồm số nguyên không âm, `+ - * /` và ngoặc thành danh sách token hậu tố. `*` `/` ưu tiên hơn `+` `-`; cùng mức thì kết hợp trái.

```python
to_rpn("3+4*2")      -> ["3", "4", "2", "*", "+"]
to_rpn("(3+4)*2")    -> ["3", "4", "+", "2", "*"]
to_rpn("10-2-3")     -> ["10", "2", "-", "3", "-"]
```

In [14]:
def exercise_3_to_rpn(s: str) -> List[str]:
    # TODO
    pass


# assert exercise_3_to_rpn("3+4*2") == ["3", "4", "2", "*", "+"]
# assert exercise_3_to_rpn("(3+4)*2") == ["3", "4", "+", "2", "*"]
# assert exercise_3_to_rpn("10-2-3") == ["10", "2", "-", "3", "-"]

## Bài 4 — Biến thể: Basic Calculator II (LeetCode #227)

Tính biểu thức trung tố gồm số nguyên không âm, `+ - * /` và dấu cách (không có ngoặc). Chia làm tròn về 0. Làm trong **một lượt** với ngăn xếp: `+x` push `x`, `-x` push `−x`, `*`/`/` gộp với đỉnh; cuối cùng cộng ngăn xếp.

In [15]:
def exercise_4_calculate(s: str) -> int:
    # TODO
    pass


# assert exercise_4_calculate("3+2*2") == 7
# assert exercise_4_calculate(" 3/2 ") == 1
# assert exercise_4_calculate(" 3+5 / 2 ") == 5
# assert exercise_4_calculate("14-3/2") == 13

## Bài 5 — Mini challenge: hậu tố → trung tố với ngoặc tối thiểu

Cho tokens hậu tố, trả về chuỗi trung tố **chỉ thêm ngoặc khi cần** (đọc lại theo quy ước thông thường phải ra **cùng giá trị**, với chia làm tròn về 0). Quy tắc:

- toán hạng trái cần ngoặc nếu độ ưu tiên của nó **nhỏ hơn** toán tử hiện tại;
- toán hạng phải cần ngoặc nếu độ ưu tiên **nhỏ hơn**, hoặc **bằng** khi toán tử hiện tại **không phải** `+`.

Vì sao cả `*` cũng cần? Do chia làm tròn: `a*(b/c)` ≠ `a*b/c` (vd `157*(16/94) = 0` nhưng `157*16/94 = 26`). Chỉ `+` là an toàn: `a+(b-c) = a+b-c` và `a+(b+c) = a+b+c` với số nguyên.

```python
to_infix(["2","1","+","3","*"])        -> "(2+1)*3"
to_infix(["1","2","3","-","-"])        -> "1-(2-3)"
to_infix(["1","2","-","3","-"])        -> "1-2-3"
```

In [16]:
def exercise_5_to_infix(tokens: List[str]) -> str:
    # TODO
    pass


# assert exercise_5_to_infix(["2", "1", "+", "3", "*"]) == "(2+1)*3"
# assert exercise_5_to_infix(["1", "2", "3", "-", "-"]) == "1-(2-3)"
# assert exercise_5_to_infix(["1", "2", "-", "3", "-"]) == "1-2-3"
# assert exercise_5_to_infix(["4", "13", "5", "/", "+"]) == "4+13/5"
# assert exercise_5_to_infix(["7", "6", "4", "/", "*"]) == "7*(6/4)"
# assert exercise_5_to_infix(["1", "2", "3", "-", "+"]) == "1+2-3"

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_rpn` sau khi tự làm.

**Bài 3**
- Mức 1: duyệt ký tự; số (có thể nhiều chữ số) → đưa thẳng vào output.
- Mức 2: toán tử `o`: trong khi đỉnh ngăn xếp là toán tử có ưu tiên `>=` `o` → pop ra output; rồi push `o`.
- Mức 3: `(` → push; `)` → pop ra output tới khi gặp `(` (bỏ `(`). Cuối cùng pop hết.

**Bài 4**
- Mức 1: giữ `num` và `sign` (toán tử **trước** `num`). Khi gặp toán tử mới hoặc hết chuỗi → xử lý `num` theo `sign`.

**Bài 5**
- Mức 1: ngăn xếp chứa cặp `(chuỗi, độ ưu tiên)`; số có ưu tiên vô cùng.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
print("Kết quả:", trace_rpn(["5", "1", "2", "+", "4", "*", "+", "3", "-"]))

 token | stack
     5 | [5]
     1 | [5, 1]
     2 | [5, 1, 2]
     + | [5, 3]   (1 + 2 = 3)
     4 | [5, 3, 4]
     * | [5, 12]   (3 * 4 = 12)
     + | [17]   (5 + 12 = 17)
     3 | [17, 3]
     - | [14]   (17 - 3 = 14)
Kết quả: 14


## Đáp án Bài 2

In [18]:
def answer_2_eval_rpn(tokens: List[str]) -> int:
    return Solution().evalRPN(tokens)

assert answer_2_eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert answer_2_eval_rpn(["4", "13", "5", "/", "+"]) == 6
assert answer_2_eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)

## Đáp án Bài 3

In [19]:
PREC = {"+": 1, "-": 1, "*": 2, "/": 2}

def answer_3_to_rpn(s: str) -> List[str]:
    out, ops, i = [], [], 0
    while i < len(s):
        c = s[i]
        if c.isdigit():
            j = i
            while j < len(s) and s[j].isdigit():
                j += 1
            out.append(s[i:j]); i = j
            continue
        if c in PREC:
            while ops and ops[-1] in PREC and PREC[ops[-1]] >= PREC[c]:
                out.append(ops.pop())
            ops.append(c)
        elif c == "(":
            ops.append(c)
        elif c == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()
        i += 1
    while ops:
        out.append(ops.pop())
    return out

assert answer_3_to_rpn("3+4*2") == ["3", "4", "2", "*", "+"]
assert answer_3_to_rpn("(3+4)*2") == ["3", "4", "+", "2", "*"]
assert answer_3_to_rpn("10-2-3") == ["10", "2", "-", "3", "-"]
print(answer_3_to_rpn("(1+2)*(3-4)/5"), "=", eval_rpn(answer_3_to_rpn("(1+2)*(3-4)/5")))
print("Bài 3 PASS")

['1', '2', '+', '3', '4', '-', '*', '5', '/'] = 0
Bài 3 PASS


## Đáp án Bài 4

In [20]:
def answer_4_calculate(s: str) -> int:
    stack, num, sign = [], 0, "+"
    for i, c in enumerate(s):
        if c.isdigit():
            num = num * 10 + int(c)
        if (c in "+-*/") or i == len(s) - 1:
            if sign == "+":
                stack.append(num)
            elif sign == "-":
                stack.append(-num)
            elif sign == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(div_trunc(stack.pop(), num))
            sign, num = c, 0
    return sum(stack)

assert answer_4_calculate("3+2*2") == 7
assert answer_4_calculate(" 3/2 ") == 1
assert answer_4_calculate(" 3+5 / 2 ") == 5
assert answer_4_calculate("14-3/2") == 13
for _ in range(3000):
    n = random.randint(1, 6)
    nums = [random.randint(0, 30) for _ in range(n)]
    ops_ = [random.choice("+-*/") for _ in range(n - 1)]
    for k in range(n - 1):
        if ops_[k] == "/" and nums[k + 1] == 0:
            nums[k + 1] = 1
    expr = str(nums[0]) + "".join(o + str(x) for o, x in zip(ops_, nums[1:]))
    assert answer_4_calculate(expr) == eval_rpn(answer_3_to_rpn(expr)), expr
print("Bài 4 PASS (đối chiếu với shunting-yard + evalRPN)")

Bài 4 PASS (đối chiếu với shunting-yard + evalRPN)


Chú ý: điều kiện `i == len(s) - 1` để xử lý số cuối dù chuỗi kết thúc bằng dấu cách (vì dấu cách không phải số, `num` vẫn giữ giá trị và được xử lý ở ký tự cuối).

## Đáp án Bài 5

In [21]:
def answer_5_to_infix(tokens: List[str]) -> str:
    INF = 10
    st = []                                    # (chuỗi, độ ưu tiên)
    for t in tokens:
        if t in PREC:
            b, pb = st.pop()
            a, pa = st.pop()
            p = PREC[t]
            if pa < p:
                a = f"({a})"
            if pb < p or (pb == p and t != "+"):
                b = f"({b})"
            st.append((f"{a}{t}{b}", p))
        else:
            st.append((t, INF))
    return st[0][0]

assert answer_5_to_infix(["2", "1", "+", "3", "*"]) == "(2+1)*3"
assert answer_5_to_infix(["1", "2", "3", "-", "-"]) == "1-(2-3)"
assert answer_5_to_infix(["1", "2", "-", "3", "-"]) == "1-2-3"
assert answer_5_to_infix(["4", "13", "5", "/", "+"]) == "4+13/5"
assert answer_5_to_infix(["7", "6", "4", "/", "*"]) == "7*(6/4)"
assert answer_5_to_infix(["1", "2", "3", "-", "+"]) == "1+2-3"

cnt = 0
for _ in range(5000):
    toks, val = random_expr(random.randint(0, 5))
    toks = [t if t in PREC else str(abs(int(t))) for t in toks]
    try:
        v = eval_rpn(toks)
    except ZeroDivisionError:
        continue
    infix = answer_5_to_infix(toks)
    assert eval_rpn(answer_3_to_rpn(infix)) == v, (toks, infix)     # đọc lại trung tố phải ra cùng giá trị
    cnt += 1
print(f"Bài 5 PASS ({cnt} biểu thức: hậu tố → trung tố → tính lại khớp giá trị)")
print(answer_5_to_infix(["157", "16", "94", "/", "*"]), "=", 157 * (16 // 94), "| bỏ ngoặc:", "157*16/94 =", 157 * 16 // 94)

Bài 5 PASS (4989 biểu thức: hậu tố → trung tố → tính lại khớp giá trị)
157*(16/94) = 0 | bỏ ngoặc: 157*16/94 = 26


Kiểm tra vòng: hậu tố → trung tố → shunting-yard → tính lại phải ra **cùng giá trị**. Khi viết bài, phiên bản đầu chỉ ngoặc toán hạng phải với `-` và `/` — bộ kiểm tra ngẫu nhiên bắt được ngay phản ví dụ kiểu `157*(16/94)`: bỏ ngoặc thì giá trị đổi từ `0` thành `26`.

# 14. Tổng kết

> RPN: duyệt token, số → push; toán tử → `b = pop()`, `a = pop()`, push `a op b`. Chia **làm tròn về 0** (`int(a / b)`), không dùng `//`. Nhận diện toán tử bằng tập `{+, -, *, /}` vì có số âm như `"-11"`. `O(n)` thời gian và bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Pop `b` trước, `a` sau, tính `a op b`?
- [ ] Chia làm tròn về 0?
- [ ] `"-11"` được hiểu là số?
- [ ] Trả `stack[0]` (hoặc `stack[-1]`)?